# Lab 1.1: Explicit Criteria vs Vague Instructions

In this notebook you'll run two prompts against the same buggy code and see — concretely — why explicit criteria crush vague instructions for precision.

**What you'll learn:**
- Why "be conservative" doesn't reduce false positives
- How to write categorical review criteria
- The difference between imprecise comments and contradictory comments

**The real bug in the code below:** The `validate_payment` comment says "Only accepts credit cards" but the code accepts credit_card, debit_card, AND apple_pay. That's a direct contradiction.

**The traps:** Several other comments are slightly imprecise but NOT wrong. A good review prompt catches the real bug and skips the traps.

## Setup

In [4]:
from dotenv import load_dotenv
import os

# load_dotenv() searches cwd and parents for .env
if load_dotenv():
    print("✅ .env loaded")
elif load_dotenv(os.path.expanduser("~/Documents/GitHub/caludeArchitect/.env")):
    print("✅ .env loaded (absolute path)")
else:
    print("⚠️  Could not find .env — set ANTHROPIC_API_KEY manually")

✅ .env loaded


In [5]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

## The code under review

Read this carefully. Identify which comments are **contradicted by the code** vs which are merely **imprecise but acceptable**.

In [6]:
CODE_UNDER_REVIEW = '''
class OrderProcessor:
    """Processes customer orders and handles payment validation."""
    
    def calculate_total(self, items):
        # Returns the sum of all item prices (tax included)
        subtotal = sum(item.price * item.quantity for item in items)
        tax = subtotal * 0.08  # 8% tax rate
        return subtotal + tax
    
    def validate_payment(self, amount, payment_method):
        # Only accepts credit cards
        if payment_method in ("credit_card", "debit_card", "apple_pay"):
            return self._charge(amount, payment_method)
        return False
    
    def apply_discount(self, total, code):
        # Applies percentage-based discount codes
        discount = self.lookup_discount(code)
        if discount and discount.type == "percentage":
            return total * (1 - discount.value / 100)
        elif discount and discount.type == "fixed":
            return total - discount.value
        return total
    
    def get_shipping_estimate(self, address):
        # Calculates shipping cost based on distance
        zone = self.get_zone(address)
        weight = self.get_total_weight()
        # Standard shipping: 3-5 business days
        return self.shipping_rates[zone] * weight

    def process_order(self, order):
        # Validates inventory before processing
        total = self.calculate_total(order.items)
        total = self.apply_discount(total, order.discount_code)
        payment_ok = self.validate_payment(total, order.payment)
        if payment_ok:
            self.update_inventory(order.items)  
            self.send_confirmation(order.customer)
        return payment_ok
'''

## Ground truth

Before running any prompts, let's establish what a perfect review would find.

In [7]:
# === GROUND TRUTH ===

REAL_BUGS = {
    "validate_payment": {
        "comment": "Only accepts credit cards",
        "reality": "Code accepts credit_card, debit_card, and apple_pay",
        "verdict": "TRUE BUG — direct contradiction between comment and code"
    }
}

FALSE_POSITIVE_TRAPS = {
    "calculate_total": {
        "comment": "Returns the sum of all item prices (tax included)",
        "why_its_ok": "The method DOES return a tax-included total. 'tax included' describes the output, not the input. Imprecise but not contradictory."
    },
    "apply_discount": {
        "comment": "Applies percentage-based discount codes",
        "why_its_ok": "The code handles both percentage and fixed discounts. The comment describes partial behavior but doesn't claim fixed discounts DON'T work. Incomplete ≠ contradictory."
    },
    "get_shipping_estimate": {
        "comment": "Calculates shipping cost based on distance",
        "why_its_ok": "The code uses zone lookup (which correlates with distance) and weight. Saying 'based on distance' is an oversimplification, not a contradiction."
    },
    "process_order": {
        "comment": "Validates inventory before processing",
        "why_its_ok": "The method doesn't explicitly validate inventory — but it calls update_inventory which may do validation internally. The comment describes intent/design, not a contradiction."
    }
}

print("=== REAL BUGS (should be caught) ===")
for func, info in REAL_BUGS.items():
    print(f"  {func}: {info['verdict']}")

print(f"\n=== FALSE POSITIVE TRAPS (should be skipped) ===")
for func, info in FALSE_POSITIVE_TRAPS.items():
    print(f"  {func}: {info['why_its_ok']}")

=== REAL BUGS (should be caught) ===
  validate_payment: TRUE BUG — direct contradiction between comment and code

=== FALSE POSITIVE TRAPS (should be skipped) ===
  calculate_total: The method DOES return a tax-included total. 'tax included' describes the output, not the input. Imprecise but not contradictory.
  apply_discount: The code handles both percentage and fixed discounts. The comment describes partial behavior but doesn't claim fixed discounts DON'T work. Incomplete ≠ contradictory.
  get_shipping_estimate: The code uses zone lookup (which correlates with distance) and weight. Saying 'based on distance' is an oversimplification, not a contradiction.
  process_order: The method doesn't explicitly validate inventory — but it calls update_inventory which may do validation internally. The comment describes intent/design, not a contradiction.


---
## Experiment 1: The vague prompt

This is the kind of prompt most people write first. It sounds reasonable but produces unreliable results.

In [8]:
VAGUE_PROMPT = f"""You are a code reviewer. Review the following code and check that 
all comments are accurate. Report any issues you find.

Be conservative and only report high-confidence findings.

Code:
```python
{CODE_UNDER_REVIEW}
```

Return your findings as a JSON array. Each finding should have:
- "location": function name
- "issue": description of the problem
- "severity": "high", "medium", or "low"

Return ONLY the JSON array, no other text."""

print("=== VAGUE PROMPT ===")
print(VAGUE_PROMPT[:200] + "...")

=== VAGUE PROMPT ===
You are a code reviewer. Review the following code and check that 
all comments are accurate. Report any issues you find.

Be conservative and only report high-confidence findings.

Code:
```python

c...


In [9]:
def run_review(prompt, label=""):
    """Run a review prompt and parse the JSON response."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    
    # Clean up common formatting issues
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    try:
        findings = json.loads(raw)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON. Raw response:\n{raw}")
        return []
    
    return findings

In [10]:
# Run the vague prompt
vague_findings = run_review(VAGUE_PROMPT)

print(f"Vague prompt reported {len(vague_findings)} findings:\n")
for i, f in enumerate(vague_findings, 1):
    loc = f.get('location', 'unknown')
    is_real = loc in REAL_BUGS
    tag = "✅ TRUE BUG" if is_real else "❌ FALSE POSITIVE"
    print(f"  {i}. [{f.get('severity', '?').upper()}] {loc}: {f.get('issue', '')}")
    print(f"     → {tag}")
    if not is_real and loc in FALSE_POSITIVE_TRAPS:
        print(f"     → Why it's a trap: {FALSE_POSITIVE_TRAPS[loc]['why_its_ok']}")
    print()

Vague prompt reported 4 findings:

  1. [MEDIUM] validate_payment: Comment says 'Only accepts credit cards' but code accepts credit_card, debit_card, and apple_pay
     → ✅ TRUE BUG

  2. [MEDIUM] apply_discount: Comment says 'Applies percentage-based discount codes' but code handles both percentage and fixed discount types
     → ❌ FALSE POSITIVE
     → Why it's a trap: The code handles both percentage and fixed discounts. The comment describes partial behavior but doesn't claim fixed discounts DON'T work. Incomplete ≠ contradictory.

  3. [MEDIUM] get_shipping_estimate: Comment says 'Calculates shipping cost based on distance' but code calculates based on zone and weight, not distance
     → ❌ FALSE POSITIVE
     → Why it's a trap: The code uses zone lookup (which correlates with distance) and weight. Saying 'based on distance' is an oversimplification, not a contradiction.

  4. [HIGH] process_order: Comment says 'Validates inventory before processing' but inventory validation is no

In [11]:
# Score it
vague_true_pos = sum(1 for f in vague_findings if f.get('location') in REAL_BUGS)
vague_false_pos = sum(1 for f in vague_findings if f.get('location') not in REAL_BUGS)
vague_precision = vague_true_pos / max(len(vague_findings), 1)

print(f"=== VAGUE PROMPT SCORECARD ===")
print(f"  True positives:  {vague_true_pos} / {len(REAL_BUGS)}")
print(f"  False positives: {vague_false_pos}")
print(f"  Precision:       {vague_precision:.0%}")
print(f"")
if vague_false_pos > 0:
    print(f"  ⚠️  {vague_false_pos} false positive(s) — this is what erodes developer trust.")
    print(f"  Notice: 'be conservative' did NOT prevent these.")

=== VAGUE PROMPT SCORECARD ===
  True positives:  1 / 1
  False positives: 3
  Precision:       25%

  ⚠️  3 false positive(s) — this is what erodes developer trust.
  Notice: 'be conservative' did NOT prevent these.


---
## Experiment 2: The precise prompt

Same code, same task — but with explicit categorical criteria defining what counts as a finding.

In [ ]:
PRECISE_PROMPT = f"""You are a code reviewer. Review the following code comments 
against actual code behavior.

REVIEW CRITERIA — flag a comment ONLY when:
1. The comment claims the code does X, but the code actually does Y 
   (direct contradiction between stated and actual behavior)
2. The comment states a constraint that the code does not enforce
   (e.g., comment says "only accepts X" but code also accepts Y and Z)

DO NOT flag:
- Comments that are slightly imprecise but not misleading 
  (e.g., describing partial behavior is acceptable if it doesn't claim 
  other behavior doesn't exist)
- Comments about future behavior or design intent
- Missing comments or undocumented code
- Minor terminology differences (e.g., "cost" vs "price")
- Incomplete descriptions that omit some functionality but don't 
  contradict the functionality they describe

Code:
```python
{CODE_UNDER_REVIEW}
```

Return your findings as a JSON array. Each finding should have:
- "location": function name
- "issue": description of the contradiction
- "severity": "high", "medium", or "low"
- "comment_claims": what the comment says
- "code_does": what the code actually does

Return ONLY the JSON array, no other text.
If no comments meet the criteria, return an empty array []."""

print("=== PRECISE PROMPT ===")
print("Key difference: explicit DO/DON'T criteria instead of 'be conservative'")

In [ ]:
# Run the precise prompt
precise_findings = run_review(PRECISE_PROMPT)

print(f"Precise prompt reported {len(precise_findings)} findings:\n")
for i, f in enumerate(precise_findings, 1):
    loc = f.get('location', 'unknown')
    is_real = loc in REAL_BUGS
    tag = "✅ TRUE BUG" if is_real else "❌ FALSE POSITIVE"
    print(f"  {i}. [{f.get('severity', '?').upper()}] {loc}")
    print(f"     Comment claims: {f.get('comment_claims', 'N/A')}")
    print(f"     Code actually:  {f.get('code_does', 'N/A')}")
    print(f"     → {tag}")
    print()

In [ ]:
# Score and compare
precise_true_pos = sum(1 for f in precise_findings if f.get('location') in REAL_BUGS)
precise_false_pos = sum(1 for f in precise_findings if f.get('location') not in REAL_BUGS)
precise_precision = precise_true_pos / max(len(precise_findings), 1)

print("=" * 50)
print("COMPARISON: VAGUE vs PRECISE")
print("=" * 50)
print(f"{'Metric':<25} {'Vague':>10} {'Precise':>10}")
print(f"{'-'*25} {'-'*10} {'-'*10}")
print(f"{'Total findings':<25} {len(vague_findings):>10} {len(precise_findings):>10}")
print(f"{'True positives':<25} {vague_true_pos:>10} {precise_true_pos:>10}")
print(f"{'False positives':<25} {vague_false_pos:>10} {precise_false_pos:>10}")
print(f"{'Precision':<25} {vague_precision:>9.0%} {precise_precision:>9.0%}")
print()
if precise_false_pos < vague_false_pos:
    print("🎯 Explicit criteria reduced false positives without losing the real bug.")
    print("   This is the pattern the exam tests: categorical rules > confidence filtering.")
elif precise_false_pos == 0 and precise_true_pos > 0:
    print("🎯 Perfect score! The precise prompt found only real bugs.")

---
## Experiment 3: Prove that "high-confidence" is unreliable

The exam loves using "only report high-confidence findings" as a distractor answer. Let's prove it doesn't work by running it 5 times and checking consistency.

In [ ]:
print("Running vague prompt 5 times to check consistency...\n")

results = []
for run in range(5):
    findings = run_review(VAGUE_PROMPT)
    locations = sorted(f.get('location', '?') for f in findings)
    fp = sum(1 for f in findings if f.get('location') not in REAL_BUGS)
    tp = sum(1 for f in findings if f.get('location') in REAL_BUGS)
    results.append({"run": run + 1, "findings": len(findings), "tp": tp, "fp": fp, "locations": locations})
    print(f"  Run {run+1}: {len(findings)} findings (TP={tp}, FP={fp}) — {locations}")

print(f"\n=== CONSISTENCY CHECK ===")
fp_counts = [r['fp'] for r in results]
print(f"False positives across 5 runs: {fp_counts}")
print(f"Min FP: {min(fp_counts)}, Max FP: {max(fp_counts)}")
if max(fp_counts) > min(fp_counts):
    print(f"\n⚠️  Inconsistent! 'Be conservative' produces different false positive counts")
    print(f"   each run. This is why it's unreliable in production.")
print(f"\nThe exam takeaway: 'be conservative' is NOT a precision control.")
print(f"Explicit categorical criteria (precise prompt) give you deterministic behavior.")

---
## Exercise: Write your own criteria

Here's a different code snippet with its own set of comment issues. Write a precise review prompt that catches the real bugs and skips the traps.

**Your goal:** 100% recall (catch all real bugs), 0 false positives.

In [ ]:
EXERCISE_CODE = '''
class UserManager:
    """Manages user accounts and authentication."""
    
    def create_user(self, email, password):
        # Creates a new user with email verification
        hashed = self.hash_password(password)
        user = User(email=email, password_hash=hashed, verified=False)
        self.db.save(user)
        return user  # Note: verification email sent by a separate cron job
    
    def authenticate(self, email, password):
        # Returns True only for verified users
        user = self.db.find_by_email(email)
        if user and self.check_password(password, user.password_hash):
            return user  # Returns user object regardless of verified status
        return None
    
    def reset_password(self, email):
        # Sends a password reset link via email
        user = self.db.find_by_email(email)
        if user:
            token = self.generate_reset_token(user)
            self.email_service.send_reset(email, token)
            return True
        return True  # Returns True even if user not found (prevents enumeration)
    
    def delete_user(self, user_id):
        # Permanently deletes user data
        user = self.db.find_by_id(user_id)
        if user:
            user.status = "deleted"
            user.email = f"deleted_{user_id}@placeholder.com"
            self.db.save(user)  # Soft delete — anonymizes, doesn't remove
'''

EXERCISE_GROUND_TRUTH = {
    "real_bugs": {
        "authenticate": "Comment says 'Returns True only for verified users' but code returns user object for ANY matching user regardless of verified status",
        "delete_user": "Comment says 'Permanently deletes user data' but code does a soft delete (sets status to deleted, anonymizes email)"
    },
    "traps": {
        "create_user": "Comment says 'with email verification' — verification IS part of the flow (via cron), just not in this method. Design intent, not contradiction.",
        "reset_password": "Returning True for non-existent users is a deliberate security pattern (prevents user enumeration), documented in the inline comment."
    }
}

print("Code loaded. Two real bugs, two traps.")
print("Write your prompt in the next cell.")

In [ ]:
# ============================================================
# YOUR TURN: Write a precise review prompt for the code above
# ============================================================
#
# Requirements:
# - Define explicit DO flag / DON'T flag criteria
# - Should catch: authenticate (verified check missing) and 
#   delete_user (soft delete, not permanent)
# - Should skip: create_user (verification IS part of flow) 
#   and reset_password (deliberate security pattern)
#
# Hint: think about what distinguishes a contradiction from 
# a design description or a security pattern.

YOUR_PROMPT = f"""YOUR PROMPT HERE

Code:
```python
{EXERCISE_CODE}
```

Return ONLY a JSON array with objects containing: location, issue, 
severity, comment_claims, code_does.
If nothing meets criteria, return [].
"""

# Uncomment below when you've written your prompt:
# your_findings = run_review(YOUR_PROMPT)
# 
# print(f"Your prompt found {len(your_findings)} issues:\n")
# for f in your_findings:
#     loc = f.get('location', '?')
#     is_real = loc in EXERCISE_GROUND_TRUTH['real_bugs']
#     tag = "✅ CORRECT" if is_real else "❌ FALSE POSITIVE"
#     print(f"  {loc}: {tag}")
#     if not is_real and loc in EXERCISE_GROUND_TRUTH['traps']:
#         print(f"    Trap: {EXERCISE_GROUND_TRUTH['traps'][loc]}")

---
## Exam-style question

Test yourself before moving on.

In [ ]:
print("""
SCENARIO: Your CI pipeline uses Claude to review pull requests. 
Developers report that the comment-accuracy check has a 45% false 
positive rate. Other categories (null safety, error handling) are 
accurate at 92%. Developer trust in the entire system is declining.

What is the most effective first step?

A) Add "only flag high-confidence comment issues" to the review prompt

B) Temporarily disable the comment-accuracy category and rebuild its
   prompt with explicit contradiction criteria and few-shot examples

C) Increase model temperature for more diverse review approaches

D) Switch to a larger model with better reasoning capabilities
""")

# your_answer = "?"  # Set to A, B, C, or D

# Uncomment to check:
# correct = "B"
# if your_answer == correct:
#     print("✅ Correct! Disable the noisy category to restore trust immediately,")
#     print("   then rebuild its prompt with explicit criteria.")
#     print("   A fails because 'high-confidence' doesn't improve precision.")
#     print("   C is nonsensical. D doesn't address prompt quality.")
# else:
#     print(f"❌ Your answer: {your_answer}. Correct: {correct}")
#     print("   Review the unit-1.1 markdown for why.")

---
## Key takeaways

1. **"Be conservative" is not a precision control.** It produces inconsistent results across runs.
2. **Explicit categorical criteria** (flag X, don't flag Y) give deterministic precision improvement.
3. **The distinction is: contradiction vs imprecision.** Comments that describe partial behavior are acceptable. Comments that claim the opposite of what the code does are bugs.
4. **On the exam**, when a question asks how to reduce false positives, look for the answer with specific criteria — never the one adding confidence filtering.

## Next

→ `02-few-shot-extraction.ipynb` — Unit 1.2: Few-shot prompting for consistency